# EDHReader Demo

Demo for [`EDHReader`](https://github.com/diyclassics/latincy-readers), a reader for EpiDoc TEI-XML files from the [Epigraphic Database Heidelberg](https://github.com/epigraphic-database-heidelberg/data) — ~82,000 Latin inscriptions from across the Roman Empire (CC-BY-SA 4.0).

Leiden-convention markup is normalized automatically: abbreviations expanded (`D(is)` → `Dis`), editor restorations included, erasures dropped. Greek-only inscriptions are silently skipped.

## Setup

In [1]:
from latincyreaders import EDHReader, AnnotationLevel
from pathlib import Path
from pprint import pprint
from collections import Counter

In [2]:
# Local clone of https://github.com/epigraphic-database-heidelberg/data
# git clone --depth 1 https://github.com/epigraphic-database-heidelberg/data ~/latincy_data/edh-data
EDH_PATH = Path.home() / "latincy_data" / "edh-data"

reader = EDHReader(root=EDH_PATH, annotation_level=AnnotationLevel.NONE)

## File Discovery

In [3]:
reader.fileids()[:8]

['inscriptions/HD000001.xml', 'inscriptions/HD000002.xml']

In [4]:
len(reader.fileids())

2

## Metadata

`headers()` returns metadata for Latin inscriptions only (Greek-only files are skipped). No NLP processing required.

In [5]:
list(reader.headers())[:5]

[{'filename': 'HD000001.xml',
  'path': '/Volumes/fiona/work/code/diy/latincy-v3/latincy-readers/tests/fixtures/edh/inscriptions/HD000001.xml',
  'hd_nr': 'HD000001',
  'not_before': '71',
  'not_after': '130',
  'province': 'Latium et Campania (Regio I)',
  'type_of_inscription': 'epitaph'}]

In [6]:
# Latin inscriptions vs total files
all_files = reader.fileids()
latin_headers = list(reader.headers())

print(f"Total XML files : {len(all_files):,}")
print(f"Latin inscriptions: {len(latin_headers):,}")
print(f"Skipped (Greek/other): {len(all_files) - len(latin_headers):,}")

Total XML files : 2
Latin inscriptions: 1
Skipped (Greek/other): 1


## Core Interface

### texts()

In [7]:
# Raw NLP-ready text: abbreviations expanded, markup stripped
next(reader.texts())

'Dis Manibus\nNoniae Publi filiae Optatae\nGaio Iulio Artemoni\nparentibus pientissimis'

### docs()

In [8]:
reader_nlp = EDHReader(root=EDH_PATH)
doc = next(reader_nlp.docs())
doc

Dis Manibus Noniae Publi filiae Optatae Gaio Iulio Artemoni parentibus pientissimis

In [9]:
pprint(doc._.metadata)

{'filename': 'HD000001.xml',
 'hd_nr': 'HD000001',
 'lines': [(1, 'Dis Manibus'),
           (2, 'Noniae Publi filiae Optatae'),
           (3, 'Gaio Iulio Artemoni'),
           (4, 'parentibus pientissimis')],
 'not_after': '130',
 'not_before': '71',
 'path': '/Volumes/fiona/work/code/diy/latincy-v3/latincy-readers/tests/fixtures/edh/inscriptions/HD000001.xml',
 'province': 'Latium et Campania (Regio I)',
 'type_of_inscription': 'epitaph'}


### sents()

In [10]:
list(reader_nlp.sents())[:8]

[Dis Manibus Noniae Publi filiae Optatae Gaio Iulio Artemoni parentibus pientissimis]

### tokens()

In [11]:
list(reader_nlp.tokens())[:8]

[Dis, Manibus, Noniae, Publi, filiae, Optatae, Gaio, Iulio]

In [12]:
tok = next(reader_nlp.tokens())
tok.text, tok.lemma_, tok.pos_

('Dis', 'deus', 'NOUN')

## EDH Features

### Line spans and citations

Each inscription line is tracked as a named span with citation key `HD000001.N`.

In [13]:
for line in doc.spans["lines"]:
    print(f"{line._.citation}: {line.text}")

HD000001.1: Dis Manibus
HD000001.2: Noniae Publi filiae Optatae
HD000001.3: Gaio Iulio Artemoni
HD000001.4: parentibus pientissimis


### Leiden abbreviation expansion

`<expan><abbr>D</abbr><ex>is</ex></expan>` → `Dis` — the expanded form is used for NLP; diplomatic (unexpanded) form is not stored.

In [14]:
# Line 1 of the first inscription: D(is) M(anibus)
line1 = next(s for s in doc.spans["lines"] if s._.citation.endswith(".1"))
print(f"Citation : {line1._.citation}")
print(f"Text     : {line1.text}")
print(f"Tokens   : {[t.text for t in line1]}")

Citation : HD000001.1
Text     : Dis Manibus
Tokens   : ['Dis', 'Manibus']


### Inscription metadata

In [ ]:
# Key fields available per inscription
h = list(reader.headers())[0]
print(f"HD number          : {h['hd_nr']}")
print(f"Date range (CE)    : {h['not_before']}–{h['not_after']}")
print(f"Province           : {h['province']}")
print(f"Type of inscription: {h['type_of_inscription']}")


HD number          : HD000001
Date range (CE)    : 71–130
Province           : Latium et Campania (Regio I)
Type of inscription: epitaph


### Corpus overview: inscription types

In [16]:
type_counts: Counter = Counter()
for h in reader.headers():
    t = h.get("type_of_inscription") or "unknown"
    type_counts[t] += 1

print(f"{'Type':<35} {'Count':>6}")
print("-" * 43)
for t, n in type_counts.most_common(8):
    print(f"{t:<35} {n:>6}")
print()
print(f"Total Latin inscriptions: {sum(type_counts.values()):,}")

Type                                 Count
-------------------------------------------
epitaph                                  1

Total Latin inscriptions: 1


### Corpus overview: provinces

In [17]:
province_counts: Counter = Counter()
for h in reader.headers():
    p = h.get("province") or "unknown"
    province_counts[p] += 1

print(f"{'Province':<45} {'Count':>6}")
print("-" * 53)
for p, n in province_counts.most_common(8):
    print(f"{p:<45} {n:>6}")
print()
print(f"Total provinces: {len(province_counts)}")

Province                                       Count
-----------------------------------------------------
Latium et Campania (Regio I)                       1

Total provinces: 1


### Corpus overview: date distribution

In [18]:
# Midpoint date distribution by century
century_counts: Counter = Counter()
dated = 0
for h in reader.headers():
    nb = h.get("not_before")
    na = h.get("not_after")
    if nb and na:
        try:
            mid = (int(nb) + int(na)) / 2
            cent = int(mid // 100) * 100
            century_counts[cent] += 1
            dated += 1
        except ValueError:
            pass

print(f"Dated inscriptions: {dated:,}")
print()
print(f"{'Century (CE)':<15} {'Count':>6}")
print("-" * 23)
for cent in sorted(century_counts):
    label = f"{cent}s" if cent >= 0 else f"{abs(cent)}s BCE"
    print(f"{label:<15} {century_counts[cent]:>6}")

Dated inscriptions: 1

Century (CE)     Count
-----------------------
100s                 1


### Annotation levels

In [19]:
# TOKENIZE: fast, no full model pipeline
reader_tok = EDHReader(root=EDH_PATH, annotation_level=AnnotationLevel.TOKENIZE)
tok_doc = next(reader_tok.docs())
[(t.text, t.lemma_) for t in tok_doc[:8]]

[('Dis', ''),
 ('Manibus', ''),
 ('\n', ''),
 ('Noniae', ''),
 ('Publi', ''),
 ('filiae', ''),
 ('Optatae', ''),
 ('\n', '')]